# Quantum Vision Transformers (QVT) notebook

This notebook is a lightweight companion to the `papers/quantum_vision_transformers/` reproduction.

Goals:
- Load a QVT config.
- Build a model (MerLin photonic layers).
- Run a forward pass.
- Optionally run a tiny 1-epoch smoke training run.

Expected working directory: `papers/quantum_vision_transformers/`.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
PROJECT_DIR = HERE

# Locate the repo root by walking upwards until we find `runtime_lib/`.
REPO_ROOT = None
for parent in [HERE, *HERE.parents]:
    if (parent / "runtime_lib").is_dir() and (parent / "implementation.py").exists():
        REPO_ROOT = parent
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing implementation.py")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Project:", PROJECT_DIR)
print("Repo root:", REPO_ROOT)


In [ ]:
import merlin
import torch

print("merlin:", merlin.__file__)
print("torch:", torch.__version__)


## Load a config

The shared runner uses `configs/defaults.json` and optionally merges another JSON config.
Here we load defaults + the paper's RetinaMNIST model-A config.

In [ ]:
from runtime_lib import deep_update, load_config

defaults_path = PROJECT_DIR / "configs" / "defaults.json"
paper_cfg_path = PROJECT_DIR / "configs" / "paper" / "model_a_retina.json"

defaults = load_config(defaults_path)
paper_cfg = load_config(paper_cfg_path)

cfg = deep_update(dict(defaults), dict(paper_cfg))
print("description:", cfg.get("description"))
print("dataset:", cfg.get("dataset"))
print("model_type:", cfg.get("model_type"))
print("circuit_family:", cfg.get("circuit_family"))
print("profile:", cfg.get("profile"))


## Build a model and run a forward pass

This uses the paper-local `lib/` code directly (same code the shared runner imports).

In [ ]:
from lib.config import validate_run_config
from lib.models import QVTModel
from lib.runner import resolve_runtime_dtype

cfg = validate_run_config(cfg)
runtime_dtype = resolve_runtime_dtype(cfg)
device = torch.device(cfg.get("device", "cpu"))

torch.set_default_dtype(runtime_dtype)

# Synthetic batch: (B, C, H, W). MedMNIST images are typically 28x28.
batch = torch.rand(2, 3, int(cfg.get("img_size", 28)), int(cfg.get("img_size", 28)), device=device)

model = QVTModel(
    model_type=cfg.get("model_type", "B"),
    img_size=batch.shape[2],
    in_channels=batch.shape[1],
    patch_size=int(cfg.get("patch_size", 7)),
    embed_dim=int(cfg.get("embed_dim", 16)),
    n_layers=int(cfg.get("n_layers", 4)),
    n_classes=2,  # for smoke; real loaders set this properly
    use_cls_token=bool(cfg.get("use_cls_token", True)),
    use_pos_embed=bool(cfg.get("use_pos_embed", True)),
    image_embed_grayscale=bool(cfg.get("image_embed_grayscale", False)),
    compound_readout=str(cfg.get("compound_readout", "cross_only")),
    circuit_family=str(cfg.get("circuit_family", "generic")),
    n_regions_per_side=int(cfg.get("n_regions_per_side", 2)),
    n_patches_per_side=int(cfg.get("n_patches_per_side", 2)),
    use_rpp_attention=bool(cfg.get("use_rpp_attention", True)),
    device=device,
).to(device=device, dtype=runtime_dtype)

with torch.no_grad():
    out = model(batch)

print("runtime_dtype:", runtime_dtype)
print("output shape:", tuple(out.shape))
print("param counts:", json.dumps(model.count_trainable_params(), indent=2))


## Optional: 1-epoch smoke run (downloads MedMNIST)

This runs the same entrypoint the shared runner calls: `lib.runner.train_and_evaluate(cfg, run_dir)`.

It will:
- download MedMNIST into the shared `data_root`
- create an output directory under `outdir/`

Toggle `RUN_SMOKE = True` to execute.

In [ ]:
from datetime import datetime

from lib.runner import train_and_evaluate

RUN_SMOKE = False

if RUN_SMOKE:
    smoke_cfg = dict(cfg)
    smoke_cfg["epochs"] = 1
    smoke_cfg["batch_size"] = 8
    smoke_cfg["num_workers"] = 0
    smoke_cfg["train_subset_size"] = 64
    smoke_cfg["train_subset_seed"] = 0
    smoke_cfg["train_subset_mode"] = "random"
    smoke_cfg["precision_mode"] = "gpu_friendly"  # faster default
    smoke_cfg["dtype"] = "float32"

    run_dir = PROJECT_DIR / "outdir" / f"notebook_smoke_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)

    train_and_evaluate(smoke_cfg, run_dir)
    print("Artifacts in:", run_dir)
else:
    print("RUN_SMOKE is False; skipping.")


## CLI equivalent

From this folder, the shared runner invocation is:

```bash
python ../../implementation.py --paper quantum_vision_transformers --config configs/paper/model_a_retina.json
```


## Figure helpers

The figure code lives in `scripts/analysis/generate_figures.py`.
The tests expect a wrapper at `scripts/generate_figures.py`, which re-exports a few helpers.


In [ ]:
import importlib.util

generate_figures_path = PROJECT_DIR / "scripts" / "generate_figures.py"
spec = importlib.util.spec_from_file_location("qvt_generate_figures", generate_figures_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)

variant = module.make_variant_key("A", "generic", "full")
print("variant:", variant)
print("label:", module.pretty_model_label(variant))
